# Regularization Methods (Ridge, Lasso, Elastic Net) — Google Colab

**Goal:** Control **overfitting** in linear models by adding a **penalty** on large coefficients — compare **Ridge (L2)**, **Lasso (L1)**, and **Elastic Net (L1+L2)**.

| Example | Feature(s) (X) | Target (y) | Dataset |
|---------|----------------|------------|---------|
| **Example 1** | Level (polynomial degree 7) | Salary | `Datasets/Position_Salaries.csv` |
| **Example 2** | R&D, Admin, Marketing, State | Profit | `Datasets/50_Startups.csv` |

| Phase | Topic | Cells |
|-------|-------|-------|
| Phase 0 | Setup | Install & Imports |
| — | Algorithm Guide | L1, L2, alpha, when to use each |
| Phase 1 | Data Pre-processing | Load → Cleaning → Encoding → Split |
| Phase 2 | Algorithm | Train → Predict → Visualize → Evaluate |

> **Run:** Runtime → Run all (or Ctrl+F9)

---
# Algorithm Guide — Regularization Methods

## Why Regularization?

When a model is **too flexible** (many features, high polynomial degree), it may **memorize noise** instead of learning the true pattern.

| Problem | Symptom | Regularization Fix |
|---------|---------|-------------------|
| **Overfitting** | Low train error, high test error | Penalize large coefficients |
| **Multicollinearity** | Unstable coefficients | Ridge shrinks correlated features together |
| **Too many features** | Weak interpretability | Lasso sets some coefficients to **zero** |

## General Regularized Objective

**minimize:** Data Loss + λ · Penalty

`Loss = Σ(yᵢ − ŷᵢ)² + α · Ω(β)`

In scikit-learn, **`alpha`** plays the role of **λ** (strength of penalty).

## Without Regularization (OLS / Linear Regression)

`minimize Σ(yᵢ − ŷᵢ)²`

Standard least squares — no penalty on coefficients.

## Ridge Regression (L2 Penalty)

`minimize Σ(yᵢ − ŷᵢ)² + α · Σ βⱼ²`

| Property | Detail |
|----------|--------|
| Penalty | **Sum of squared** coefficients |
| Effect | Shrinks coefficients **toward zero** smoothly |
| Feature selection | **No** — all features stay in the model |
| Best for | Many features, multicollinearity |

## Lasso Regression (L1 Penalty)

`minimize Σ(yᵢ − ŷᵢ)² + α · Σ |βⱼ|`

| Property | Detail |
|----------|--------|
| Penalty | **Sum of absolute** coefficients |
| Effect | Can shrink some coefficients **exactly to zero** |
| Feature selection | **Yes** — automatic feature selection |
| Best for | Sparse models — only a few important features |

## Elastic Net (L1 + L2)

`minimize Σ(yᵢ − ŷᵢ)² + α · [l1_ratio · Σ|βⱼ| + (1 − l1_ratio) · Σ βⱼ²]`

| Property | Detail |
|----------|--------|
| Penalty | **Mix** of Lasso and Ridge |
| `l1_ratio=1` | Pure Lasso |
| `l1_ratio=0` | Pure Ridge |
| Best for | Correlated features + need some sparsity |

## Comparison Table

| Method | Penalty | Shrinks Coefs | Zero Coefs | Feature Selection |
|--------|---------|---------------|------------|-------------------|
| **Linear Regression** | None | No | No | No |
| **Ridge** | L2 | Yes | Rarely | No |
| **Lasso** | L1 | Yes | **Yes** | **Yes** |
| **Elastic Net** | L1 + L2 | Yes | Yes | Yes |

## Key Hyperparameter: `alpha`

| alpha | Effect |
|-------|--------|
| **Small** (→ 0) | Weak penalty — closer to OLS |
| **Large** | Strong penalty — simpler model, may underfit |

> Always tune `alpha` using cross-validation (`RidgeCV`, `LassoCV`) in production.

## Feature Scaling — Required!

Regularized models are **sensitive to feature scale**. Always apply **`StandardScaler`** before Ridge, Lasso, or Elastic Net so the penalty treats all features fairly.

## What the Student Must Remember

1. Regularization = **penalty on coefficients** to reduce overfitting.
2. **Ridge (L2)** shrinks; **Lasso (L1)** can **eliminate** features.
3. **Elastic Net** combines both — good default when features are correlated.
4. **`alpha`** controls penalty strength (higher = simpler model).
5. **Always scale features** before applying regularization.

## Phase 0 — Cell 0: Install Libraries

Google Colab usually includes most libraries. This cell ensures required packages are available.

**What this cell does:** Installs scikit-learn, pandas, matplotlib, numpy, and seaborn quietly.

In [ ]:
# Install required libraries quietly (-q hides output)
!pip install -q scikit-learn pandas matplotlib numpy seaborn

## Phase 0 — Cell 1: Import Libraries

Import libraries for preprocessing, regularized linear models, and evaluation.

**What this cell does:** Loads numpy, pandas, matplotlib, sklearn Ridge/Lasso/ElasticNet, Pipeline, and metrics.

In [ ]:
# --- Import libraries ---
import numpy as np              # Numerical operations and arrays
import pandas as pd             # Load and manipulate tabular data
import matplotlib.pyplot as plt # Create charts and plots
import seaborn as sns           # Statistical visualizations (optional styling)

from sklearn.model_selection import train_test_split       # Split data into train/test
from sklearn.impute import SimpleImputer                   # Fill missing values
from sklearn.preprocessing import PolynomialFeatures, StandardScaler  # Feature engineering & scaling
from sklearn.pipeline import Pipeline                      # Chain preprocessing + model steps
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet  # OLS + regularized models
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score  # Evaluation metrics

plt.rcParams['figure.figsize'] = (10, 6)  # Default plot size: width=10, height=6 inches
sns.set_theme(style='whitegrid')            # Clean white background with grid lines
np.random.seed(42)                          # Fix random seed for reproducible splits

print('Libraries ready')                      # Confirm all imports loaded successfully

---
# Example 1: Position Salaries — Regularization with Polynomial Features

Predict **Salary** from **Level** using **degree-7 polynomial** features. High-degree polynomials **overfit** small datasets — regularization fixes this.

| Column | Role | Description |
|--------|------|-------------|
| `Level` | Feature (X) | Job level (1–10) → expanded to x, x², …, x⁷ |
| `Salary` | Target (y) | Annual salary in USD |

**File:** `Datasets/Position_Salaries.csv`

**Models compared:** Linear Regression (no reg) · Ridge · Lasso · Elastic Net

---
# Phase 1: Data Pre-processing

Prepare the data before training — same template is reused for other algorithms.

## Example 1 — Cell 1: Load and Explore Data

Load the CSV file and perform initial exploration (head, info, describe, shape).

**What this cell does:** Reads `Datasets/Position_Salaries.csv` and displays basic statistics.

In [ ]:
# Step 1) Load dataset
dataset = pd.read_csv('Datasets/Position_Salaries.csv')  # Read CSV into a DataFrame

FEATURE_COL = 'Level'   # Independent variable (X) — job level
TARGET_COL = 'Salary'   # Dependent variable (y) — salary to predict
POLY_DEG = 7            # High degree — exposes overfitting on small data (train R² → 1.0 for OLS)

print('First 5 rows:')          # Print a label for the table below
display(dataset.head())         # Show the first 5 rows to inspect the data

print('\nDataset info:')       # Print a label for column types and null counts
dataset.info()                  # Show column names, data types, and non-null counts

print('\nStatistical summary:')  # Print a label for numeric statistics
display(dataset.describe())       # Show count, mean, std, min, max, quartiles

print(f'\nShape: {dataset.shape[0]} rows x {dataset.shape[1]} columns')  # Total rows and columns

## Example 1 — Cell 2: Data Cleaning (Handling Missing Values)

Check missing values, remove duplicates, and apply imputation if needed.

**What this cell does:** Cleans the dataset before modeling.

In [ ]:
# Step 2) Data cleaning

print('Missing values per column:')  # Print a label
print(dataset.isnull().sum())        # Count NaN values in each column

rows_before = len(dataset)                              # Store row count before cleaning
dataset = dataset.drop_duplicates().reset_index(drop=True)  # Remove duplicate rows
rows_after = len(dataset)                               # Store row count after deduplication
print(f'\nDuplicates removed: {rows_before - rows_after}')  # Show duplicate count

num_cols = dataset.select_dtypes(include=[np.number]).columns.tolist()  # Numeric columns only
imputer = SimpleImputer(missing_values=np.nan, strategy='mean')  # Fill NaN with column mean
if dataset.isnull().sum().sum() > 0:               # If missing values exist
    dataset[num_cols] = imputer.fit_transform(dataset[num_cols])  # Impute numeric columns
    print('Missing values imputed with mean')      # Confirm imputation
else:
    print('No missing values — imputer not applied')  # Skip when complete

print(f'\nRows after cleaning: {rows_after}')    # Final row count

## Example 1 — Cell 3: Categorical Data Encoding

We use `Level` (numeric). The `Position` column is text — skipped because Level already encodes rank.

**What this cell does:** Checks for categorical columns and encodes if needed.

In [ ]:
# Step 3) Categorical encoding

cat_cols = dataset.select_dtypes(include=['object', 'category']).columns.tolist()  # Find text columns
print(f'Categorical columns (not used as X): {cat_cols}')  # Position names — informational only
print(f'Feature used: {FEATURE_COL} (polynomial degree {POLY_DEG})')  # Expands Level to many powers
print('No encoding required — X is numeric.')

## Example 1 — Cell 4: Splitting the Data

Define X (Level) and y (Salary), then split 80/20.

**What this cell does:** Creates feature/target arrays and applies train_test_split.

In [ ]:
# Step 4) Train-Test split

X = dataset[[FEATURE_COL]].values  # Feature matrix: Level (2D array for sklearn)
y = dataset[TARGET_COL].values     # Target vector: Salary values

X_train, X_test, y_train, y_test = train_test_split(
    X, y,                  # Data to split
    test_size=0.2,         # 20% test, 80% train
    random_state=42        # Reproducible split
)

print(f'X_train shape: {X_train.shape}')  # Training features shape
print(f'X_test shape:  {X_test.shape}')   # Test features shape
print(f'y_train shape: {y_train.shape}')  # Training targets shape
print(f'y_test shape:  {y_test.shape}')   # Test targets shape

> **Note:** Regularization requires **feature scaling**. Each model pipeline includes `StandardScaler` after polynomial expansion.

---
# Phase 2: Regularization — Polynomial Salary Model

Train and compare four models on polynomial features.

## Example 1 — Cell 5: Build Pipelines and Train Models

Create four pipelines: **No Reg (OLS)**, **Ridge**, **Lasso**, **Elastic Net** — each with PolynomialFeatures + StandardScaler.

**What this cell does:** Defines pipelines, fits all models, and stores them in a dictionary.

In [ ]:
# Step 5) Build and train regularized polynomial models

# Four models to compare — each pipeline: PolynomialFeatures → StandardScaler → Model
models = {
    'No Reg (OLS)': Pipeline([
        ('poly', PolynomialFeatures(degree=POLY_DEG, include_bias=False)),
        ('scaler', StandardScaler()),
        ('model', LinearRegression())  # Ordinary least squares — no penalty
    ]),
    'Ridge (L2)': Pipeline([
        ('poly', PolynomialFeatures(degree=POLY_DEG, include_bias=False)),
        ('scaler', StandardScaler()),
        ('model', Ridge(alpha=10.0, random_state=42))  # L2 penalty — shrinks coefficients smoothly
    ]),
    'Lasso (L1)': Pipeline([
        ('poly', PolynomialFeatures(degree=POLY_DEG, include_bias=False)),
        ('scaler', StandardScaler()),
        ('model', Lasso(alpha=500.0, max_iter=50000, random_state=42))  # L1 — can zero out terms
    ]),
    'Elastic Net': Pipeline([
        ('poly', PolynomialFeatures(degree=POLY_DEG, include_bias=False)),
        ('scaler', StandardScaler()),
        ('model', ElasticNet(alpha=10.0, l1_ratio=0.5, max_iter=50000, random_state=42))  # L1 + L2 mix
    ])
}

fitted_models = {}  # Store trained pipelines
for name, pipe in models.items():
    pipe.fit(X_train, y_train)  # Fit preprocessing + model on training data
    fitted_models[name] = pipe    # Save fitted pipeline
    print(f'{name}: trained')   # Confirm each model is ready

## Example 1 — Cell 6: Predict

Generate predictions from all four models on train and test sets.

**What this cell does:** Calls `.predict()` on each fitted pipeline.

In [ ]:
# Step 6) Predict with all models

predictions = {}  # Dictionary to store y_pred for each model
for name, pipe in fitted_models.items():
    predictions[name] = {
        'train': pipe.predict(X_train),  # Training predictions
        'test': pipe.predict(X_test)     # Test predictions
    }

print('Sample test predictions (Level -> Salary):')  # Print a label
for i in range(len(y_test)):
    level = X_test[i][0]
    print(f'  Level={level:.0f} | Actual=${y_test[i]:,.0f}')
    for name in fitted_models:
        pred = predictions[name]['test'][i]
        print(f'    {name:15s} -> ${pred:,.0f}')

## Example 1 — Cell 7: Visualization

Plot prediction curves for all four models — see how regularization **smooths** overfitting.

**What this cell does:** Creates a 2×2 subplot comparing OLS vs Ridge vs Lasso vs Elastic Net.

In [ ]:
# Step 7) Visualization — compare prediction curves

X_plot = np.linspace(X.min(), X.max(), 300).reshape(-1, 1)  # Smooth x-axis for curves
plot_order = ['No Reg (OLS)', 'Ridge (L2)', 'Lasso (L1)', 'Elastic Net']
colors = {'No Reg (OLS)': 'crimson', 'Ridge (L2)': 'seagreen', 'Lasso (L1)': 'darkorange', 'Elastic Net': 'purple'}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))  # Four subplots
axes = axes.ravel()  # Flatten to 1D for easy indexing

for ax, name in zip(axes, plot_order):
    pipe = fitted_models[name]
    y_plot = pipe.predict(X_plot)  # Model curve on dense grid
    ax.scatter(X_train, y_train, color='blue', s=70, label='Train', zorder=3)   # Training points
    ax.scatter(X_test, y_test, color='green', s=70, label='Test', zorder=3)     # Test points
    ax.plot(X_plot, y_plot, color=colors[name], linewidth=2, label=name)        # Prediction curve
    ax.set_xlabel('Level')           # X-axis label
    ax.set_ylabel('Salary (USD)')   # Y-axis label
    ax.set_title(f'{name} (degree={POLY_DEG})')  # Subplot title
    ax.legend(fontsize=8)            # Legend

plt.suptitle('Regularization Effect — Position Salaries (Polynomial)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## Example 1 — Cell 8: Evaluation

Compare **Train vs Test R²** for all models — overfitting shows as high train R² but low test R².

**What this cell does:** Computes MAE, RMSE, R² for each model on the test set.

In [ ]:
# Step 8) Evaluation — compare all models

rows = []
for name, pipe in fitted_models.items():
    y_tr_pred = predictions[name]['train']  # Training predictions
    y_te_pred = predictions[name]['test']   # Test predictions
    rows.append({
        'Model': name,
        'Train R²': r2_score(y_train, y_tr_pred),
        'Test R²': r2_score(y_test, y_te_pred),
        'Test MAE': mean_absolute_error(y_test, y_te_pred),
        'Test RMSE': np.sqrt(mean_squared_error(y_test, y_te_pred))
    })

results = pd.DataFrame(rows)
display(results.round(4))  # Show comparison table

best_name = results.loc[results['Test R²'].idxmax(), 'Model']  # Model with highest test R²
r2 = results.loc[results['Test R²'].idxmax(), 'Test R²']       # Best test R² value
print(f'\nBest model on test set: {best_name}')
print(f'Example 1 Best Test R² = {r2:.4f}')

## Why does Regularization work for Position Salaries?

| # | Reason | Explanation |
|---|--------|-------------|
| 1 | **Polynomial overfitting** | Degree-7 on 10 rows — OLS reaches Train R² = 1.0 (memorizes every point) |
| 2 | **Ridge smooths the curve** | L2 penalty shrinks large polynomial coefficients — smoother predictions |
| 3 | **Lasso simplifies** | L1 may zero out x³ or x⁴ terms — keeps only useful powers |
| 4 | **Scaling is essential** | x⁴ terms are huge without StandardScaler — penalty would be unfair |
| 5 | **Train vs Test gap** | OLS often has Train R² ≈ 1 but worse Test R² — regularization closes the gap |

> **Summary:** On small data + high polynomial degree, **regularized models generalize better** than plain OLS.

## Understanding R² — Example 1 (Position Salaries)

Compare **Train R²** vs **Test R²** across models:

| Pattern | Meaning |
|---------|---------|
| Train R² high, Test R² low | **Overfitting** — OLS without regularization |
| Train & Test R² both reasonable | **Good generalization** — Ridge/Lasso/Elastic Net |
| Both R² low | **Underfitting** — alpha too large |

> With only **2 test samples**, focus on the **curve shape** and Train/Test gap, not only one R² number.

---
# Example 2: Startup Profit — Regularization with Multiple Features

Predict **Profit** from spending columns and **State** using Ridge, Lasso, and Elastic Net on **many encoded features**.

| Column | Role | Description |
|--------|------|-------------|
| `R&D Spend` | Feature (X₁) | Research & development budget |
| `Administration` | Feature (X₂) | Administrative expenses |
| `Marketing Spend` | Feature (X₃) | Marketing budget |
| `State` | Feature (X₄) | US state (One-Hot encoded) |
| `Profit` | Target (y) | Annual profit in USD |

**File:** `Datasets/50_Startups.csv`

## Example 2 — Cell 1: Load and Explore Data

Load the startups CSV and inspect the data.

**What this cell does:** Reads `Datasets/50_Startups.csv` and displays basic statistics.

In [ ]:
# Step 1) Load dataset
dataset = pd.read_csv('Datasets/50_Startups.csv')  # Read CSV into a DataFrame

FEATURE_COLS = ['R&D Spend', 'Administration', 'Marketing Spend']  # Numeric features
CAT_COL = 'State'                                                   # Categorical feature
TARGET_COL = 'Profit'                                               # Target variable

print('First 5 rows:')
display(dataset.head())

print('\nDataset info:')
dataset.info()

print('\nStatistical summary:')
display(dataset.describe())

print(f'\nShape: {dataset.shape[0]} rows x {dataset.shape[1]} columns')

## Example 2 — Cell 2: Data Cleaning (Handling Missing Values)

Check missing values and remove duplicates.

**What this cell does:** Cleans the dataset before modeling.

In [ ]:
# Step 2) Data cleaning

print('Missing values per column:')
print(dataset.isnull().sum())

rows_before = len(dataset)
dataset = dataset.drop_duplicates().reset_index(drop=True)
rows_after = len(dataset)
print(f'\nDuplicates removed: {rows_before - rows_after}')

num_cols = dataset.select_dtypes(include=[np.number]).columns.tolist()
imputer = SimpleImputer(missing_values=np.nan, strategy='mean')
if dataset.isnull().sum().sum() > 0:
    dataset[num_cols] = imputer.fit_transform(dataset[num_cols])
    print('Missing values imputed with mean')
else:
    print('No missing values — imputer not applied')

print(f'\nRows after cleaning: {rows_after}')

## Example 2 — Cell 3: Categorical Data Encoding

Encode **State** with **One-Hot Encoding** (drop first to avoid dummy trap).

**What this cell does:** Creates dummy columns for State.

In [ ]:
# Step 3) Categorical encoding — One-Hot for State

dataset_encoded = pd.get_dummies(dataset, columns=[CAT_COL], drop_first=True)  # State -> binary columns
feature_cols = [c for c in dataset_encoded.columns if c != TARGET_COL]  # All columns except Profit

print(f'Original columns: {list(dataset.columns)}')
print(f'Encoded feature columns ({len(feature_cols)}): {feature_cols}')

## Example 2 — Cell 4: Splitting the Data

Define X (encoded features) and y (Profit), then split 80/20.

**What this cell does:** Creates feature/target arrays and applies train_test_split.

In [ ]:
# Step 4) Train-Test split

X = dataset_encoded[feature_cols].values  # Feature matrix after encoding
y = dataset_encoded[TARGET_COL].values    # Target vector: Profit

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'X_train shape: {X_train.shape}')  # (n_train, n_features)
print(f'X_test shape:  {X_test.shape}')
print(f'y_train shape: {y_train.shape}')
print(f'y_test shape:  {y_test.shape}')

## Example 2 — Cell 5: Build Pipelines and Train Models

Train **Ridge**, **Lasso**, and **Elastic Net** with StandardScaler. Compare with OLS baseline.

**What this cell does:** Fits four scaled linear models on startup data.

In [ ]:
# Step 5) Build and train regularized models (multi-feature)

models_ex2 = {
    'No Reg (OLS)': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LinearRegression())
    ]),
    'Ridge (L2)': Pipeline([
        ('scaler', StandardScaler()),
        ('model', Ridge(alpha=1.0, random_state=42))
    ]),
    'Lasso (L1)': Pipeline([
        ('scaler', StandardScaler()),
        ('model', Lasso(alpha=100.0, max_iter=50000, random_state=42))
    ]),
    'Elastic Net': Pipeline([
        ('scaler', StandardScaler()),
        ('model', ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=50000, random_state=42))
    ])
}

fitted_ex2 = {}
for name, pipe in models_ex2.items():
    pipe.fit(X_train, y_train)
    fitted_ex2[name] = pipe
    print(f'{name}: trained')

## Example 2 — Cell 6: Predict

Predict Profit on the test set with all models.

**What this cell does:** Generates predictions and shows sample outputs.

In [ ]:
# Step 6) Predict

pred_ex2 = {}
for name, pipe in fitted_ex2.items():
    pred_ex2[name] = pipe.predict(X_test)

print('Sample predictions (Test set):')
for i in range(min(5, len(y_test))):
    print(f'  Actual Profit=${y_test[i]:,.0f}')
    for name in fitted_ex2:
        print(f'    {name:15s} -> ${pred_ex2[name][i]:,.0f}')

## Example 2 — Cell 7: Visualization

Plot **Actual vs Predicted** (best model) and **coefficient comparison** across regularization methods.

**What this cell does:** Shows Lasso feature selection via zero coefficients.

In [ ]:
# Step 7) Visualization — coefficients + actual vs predicted

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# --- Left: coefficient comparison (after scaling, from each model) ---
coef_data = {}
for name, pipe in fitted_ex2.items():
    model = pipe.named_steps['model']  # Extract linear model from pipeline
    coef_data[name] = model.coef_       # Coefficient vector

coef_df = pd.DataFrame(coef_data, index=feature_cols)  # Rows = features, cols = models
coef_df.plot(kind='barh', ax=axes[0], width=0.8)         # Horizontal bar chart
axes[0].set_xlabel('Coefficient value (scaled features)')
axes[0].set_title('Coefficient Comparison — Regularization Methods')
axes[0].axvline(0, color='black', linewidth=0.8)  # Zero line — Lasso may hit this

# --- Right: Actual vs Predicted (Ridge — typically strong on this data) ---
best_pipe = fitted_ex2['Ridge (L2)']
y_pred_ridge = best_pipe.predict(X_test)
axes[1].scatter(y_test, y_pred_ridge, color='seagreen', alpha=0.7)
min_val = min(y_test.min(), y_pred_ridge.min())
max_val = max(y_test.max(), y_pred_ridge.max())
axes[1].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect prediction')
axes[1].set_xlabel('Actual Profit (USD)')
axes[1].set_ylabel('Predicted Profit (USD)')
axes[1].set_title('Actual vs Predicted — Ridge (L2)')
axes[1].legend()

plt.tight_layout()
plt.show()

# Count non-zero coefficients (Lasso feature selection)
print('\nNon-zero coefficients per model:')
for name, pipe in fitted_ex2.items():
    coefs = pipe.named_steps['model'].coef_
    n_nonzero = int(np.sum(np.abs(coefs) > 1e-6))
    print(f'  {name:15s}: {n_nonzero} / {len(coefs)} features')

## Example 2 — Cell 8: Evaluation

Compare all models with MAE, RMSE, and R² on the test set.

**What this cell does:** Computes and displays evaluation metrics.

In [ ]:
# Step 8) Evaluation
rows = []
for name, pipe in fitted_ex2.items():
    y_pred = pipe.predict(X_test)
    rows.append({
        'Model': name,
        'Test R²': r2_score(y_test, y_pred),
        'Test MAE': mean_absolute_error(y_test, y_pred),
        'Test RMSE': np.sqrt(mean_squared_error(y_test, y_pred))
    })

results = pd.DataFrame(rows)
display(results.round(4))

best_name = results.loc[results['Test R²'].idxmax(), 'Model']
r2 = results.loc[results['Test R²'].idxmax(), 'Test R²']
print(f'\nBest model on test set: {best_name}')
print(f'Example 2 Best Test R² = {r2:.4f}')

## Why does Regularization work for Startup Profit?

| # | Reason | Explanation |
|---|--------|-------------|
| 1 | **Multiple correlated spends** | R&D and Marketing may correlate — Ridge stabilizes coefficients |
| 2 | **One-Hot State columns** | Adds extra features — Lasso can drop less useful states |
| 3 | **Feature scaling** | Spend columns have different scales — StandardScaler before penalty |
| 4 | **Coefficient shrinkage** | Prevents extreme weights on noisy features |
| 5 | **Similar R² to OLS** | On this dataset OLS already fits well — regularization adds stability |

> **Compare:** Lasso may set some State dummy coefficients to **zero** — automatic feature selection.

## Understanding R² — Example 2 (Startup Profit)

| R² Value | Meaning |
|----------|---------|
| **> 0.90** | Excellent — spending + state explain most profit variation |
| **0.70–0.90** | Good fit |
| **< 0.50** | Weak — try more features or tune alpha |

> On this dataset, **Ridge/Lasso/Elastic Net** often match OLS R² while producing **smaller, more stable** coefficients — the main benefit is **generalization**, not always higher R² on the same split.